In [ ]:
import numpy as np
import os
import nibabel as nib
import pandas as pd
import nilearn.interfaces
from spectral_analysis.helper_functions import import_mask_and_parcellation

import nitime
from nitime.timeseries import TimeSeries
from nitime import utils
import nitime.algorithms as alg
import nitime.viz
from nitime.viz import drawmatrix_channels
from nitime.analysis import CoherenceAnalyzer, MTCoherenceAnalyzer

def compute_spectra(df,denoising_strategy):
    parcel_labels, _, _, _ = import_mask_and_parcellation()

    for index, scan in df.iterrows():

        denoised_dir = 'data/denoised/'+denoising_strategy+'/' + scan.subject + '/' + scan.session + '/func/'
        output_dir = 'data/spectra/'+denoising_strategy+'/' + scan.subject + '/' + scan.session + '/func/'
        data = np.loadtxt(denoised_dir + os.path.basename(scan['preproc_filename_cifti']).replace(
            '.dtseries.nii', '_denoised_parcellated_schaefertian232.txt'))
        
        confounds_file_orig = nilearn.interfaces.fmriprep.load_confounds_utils.get_confounds_file(scan['preproc_filename_cifti'], flag_full_aroma=False)
        confounds_file = 'data/interim/confounds_'+denoising_strategy+'/'+scan.subject+'/'+scan.session+'/func/' + os.path.basename(confounds_file_orig).replace('.tsv', '_filtered.tsv')
        confounds = pd.read_csv(confounds_file, sep='\t')

        confounds_motion = confounds.filter(like='trans')
        
        data_motion = np.concat([data, confounds_motion.values], axis=1)
        data_motion_labels = np.concatenate([parcel_labels, confounds_motion.columns.values])
        
         # Create TimeSeries object

        T = TimeSeries(data_motion.T, sampling_interval=scan.tr)
        T.metadata['roi'] = data_motion_labels

        C2 = MTCoherenceAnalyzer(T)

        freqs = C2.frequencies

        for f, freq in enumerate(freqs):
            fig = drawmatrix_channels(C2.coherence[:, :, f], data_motion_labels, colorbar=True, title=f'Coherence at {freq:.3f} Hz')
            fig.savefig('figures/coherence/coherence_'+str(freq)+ '_'+scan.subject+'_'+scan.session+'_'+os.path.basename(scan['preproc_filename_cifti']).replace('.dtseries.nii', '.png'), dpi=300)
        a=7
        

        
        

if __name__ == "__main__":
    import json

    with open("config.json") as f:
        config = json.load(f)

    strategies = config["strategies"]

    # Load the DataFrame containing scan information
    df = pd.read_csv('data/func_scans_table_outliers_ses-PSI_PPLSDI.csv')
    df = df[df['task']==config["task"]]
    
    # Compute spectra
    for denoising_strategy in strategies:
        if denoising_strategy != 'high-pass-motion':
            continue
        print(f"Computing spectra for denoising strategy: {denoising_strategy}")
        compute_spectra(df, denoising_strategy)